In [172]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

words = open("../names.txt", "r").read().splitlines()

In [104]:
# 1. create table to map from char to index
chars = sorted(list(set("".join(words))))
stoi = {char: i+1 for i, char in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vacab_size = len(itos)
print(f"vocabulary size: {vacab_size} sample size {len(words)}")

vocabulary size: 27 sample size 32033


In [ ]:
# 2. build data set with specific context window
def build_dataset(words, block_size=3):
    X = []
    Y = []
    for word in words: 
        context = [0] * block_size
        for ch in word + '.':
            index = stoi[ch]
            X.append(context)
            Y.append(index)
            context = context[1:] + [index]
    return torch.tensor(X), torch.tensor(Y)

X, Y = build_dataset(words)
print(f"X shape {X.shape} Y shape {Y.shape}")

X shape torch.Size([228146, 3]) Y shape torch.Size([228146])


In [164]:
# 3. set up MLP layers and params
g = torch.Generator().manual_seed(42)
emb_size = 10
block_size = 3

# embeddings: 27 x 2
C = torch.randn((vacab_size, emb_size), generator=g)
in_dim = emb_size * block_size
hidden_dim = 200
out_dim = vacab_size

# layer 1: input dim: block_size * emb_size = 6
# w1: 6 x 100, b1: 100
w1 =  torch.randn((in_dim, hidden_dim), generator=g)
b1 = torch.randn(hidden_dim, generator=g)

# layer 2: hidden dim: 100, output dim: 27
# w2: 100 x 27, b2: 27
w2 = torch.randn((hidden_dim, out_dim), generator=g)
b2 = torch.randn(out_dim, generator=g)

parameters = [C, w1, b1, w2, b2]
for p in parameters:
    p.requires_grad = True

total_params = sum(p.nelement() for p in parameters)
print("Num params:", total_params)

Num params: 11897


In [165]:
# Set up training/val/test data sets
# 80% training, 10% eval, 10% test
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[: n1], block_size)
Xdev, Ydev = build_dataset(words[n1: n2], block_size)
Xtest, Ytest = build_dataset(words[n2:], block_size)


In [166]:
# Train MLP with learning rate finder
lr_history = []
loss_history = []
loss: torch.Tensor | None = None

lre = torch.linspace(-3, 0, 20000)
lrs = 10 ** lre
sample_size = 64
for step in range(20000):
    # 1. select a mini-batch (SGD) size=32  
    batch_idxes = torch.randint(0, Xtr.shape[0], (sample_size,))

    # 2. training data with embeddings (32, 3, 2) -> (32, 6), target y (32,)
    h_input = C[Xtr[batch_idxes]].view(-1, in_dim)
    target_y = Ytr[batch_idxes]

    # 3. hidden layer with tanh activation (32, 100)
    h_output = torch.tanh(h_input @ w1 + b1) 

    # 4. output layer [32, 27]
    logits = h_output @ w2 + b2

    # 5. compute cross entropy loss
    loss = F.cross_entropy(logits, target_y)

    # 6. loss backward
    for p in parameters:
        p.grad = None
    loss.backward()

    # 7. optimize at learning rate
    learning_rate = 0.1 if step < 10000 else 0.01#lrs[step]
    for p in parameters:
        if p.grad is not None:
            p.data -= learning_rate * p.grad

    #lr_history.append(lre[step])
    lr_history.append(learning_rate)
    loss_history.append(loss.item())

loss.item()

#plt.plot(loss_history)


2.181729793548584

In [167]:
@torch.no_grad()
def evaluate_loss(X, Y):
    h_input = C[X].view(-1, in_dim)
    h_output = torch.tanh(h_input @ w1 + b1)
    logits = h_output @ w2 + b2
    return F.cross_entropy(logits, Y).item()

print(f"Dev Loss: {evaluate_loss(Xdev, Ydev):.4f}")
print(f"Test Loss: {evaluate_loss(Xtest, Ytest):.4f}")


Dev Loss: 2.5023
Test Loss: 2.5239
